<a href="https://colab.research.google.com/github/shivamkumar359/03-rd-SEM--ASSIGNMENTS/blob/main/ML_W7_ASSIGNMENT_02_2_ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip install -q reportlab

# ============================================================
# TASK 2: EVALUATE BAGGING AND OUT-OF-BAG (OOB) ERROR
# Dataset: Housing Dataset
# Base Learner: DecisionTreeRegressor
# ============================================================

import os
import warnings
import numpy as np
import pandas as pd
import sklearn

from sklearn.ensemble import BaggingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error

from reportlab.lib import colors
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib.units import inch
from reportlab.platypus import (
    SimpleDocTemplate,
    Paragraph,
    Spacer,
    Table,
    TableStyle
)

In [ ]:
# ============================================================
# 1. CREATE THE HOUSING DATASET
# ============================================================

data = {
    "House_ID": [1, 2, 3, 4, 5, 6],
    "Length":   [15, 20, 12, 25, 18, 22],
    "Width":    [10, 15, 8, 20, 12, 18],
    "Rooms":    [3, 4, 2, 5, 3, 4],
    "Age":      [5, 2, 20, 1, 10, 8],
    "Price":    [250000, 450000, 180000, 700000, 310000, 520000]
}

df = pd.DataFrame(data)

# Exclude House_ID because it is an identifier, not a feature.
X = df[["Length", "Width", "Rooms", "Age"]].copy()
y = df["Price"].copy()

print("=" * 60)
print("TASK 2: BAGGING AND OUT-OF-BAG ERROR ANALYSIS")
print("=" * 60)

print("\nHousing Dataset:")
print(df.to_string(index=False))

print("\nDataset shape:", df.shape)
print("Number of features:", X.shape[1])
print("Number of observations:", len(X))

TASK 2: BAGGING AND OUT-OF-BAG ERROR ANALYSIS

Housing Dataset:
 House_ID  Length  Width  Rooms  Age  Price
        1      15     10      3    5 250000
        2      20     15      4    2 450000
        3      12      8      2   20 180000
        4      25     20      5    1 700000
        5      18     12      3   10 310000
        6      22     18      4    8 520000

Dataset shape: (6, 6)
Number of features: 4
Number of observations: 6


In [ ]:
# ============================================================
# 2. SPLIT DATA INTO TRAINING AND TESTING SETS
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.33,
    random_state=42
)

print("\n" + "-" * 60)
print("TRAINING AND TESTING DATA")
print("-" * 60)

print("Training observations:", len(X_train))
print("Testing observations:", len(X_test))

print("\nTraining feature shape:", X_train.shape)
print("Testing feature shape:", X_test.shape)


------------------------------------------------------------
TRAINING AND TESTING DATA
------------------------------------------------------------
Training observations: 4
Testing observations: 2

Training feature shape: (4, 4)
Testing feature shape: (2, 4)


In [ ]:
# ============================================================
# 3. CONFIGURE AND TRAIN THE BAGGING REGRESSOR
# ============================================================

# Use 100 decision trees for the bagging ensemble.
# n_jobs=1 avoids parallel execution while debugging.
# Note: oob_score is set to False because the dataset is too small
# (4 training samples), meaning some samples are selected in every bootstrap,
# causing an error during OOB evaluation.
bagging_model = BaggingRegressor(
    estimator=DecisionTreeRegressor(random_state=42),
    n_estimators=100,
    max_samples=1.0,
    max_features=1.0,
    bootstrap=True,
    bootstrap_features=False,
    oob_score=False,
    n_jobs=1,
    random_state=42
)

print("\n" + "-" * 60)
print("BAGGING MODEL CONFIGURATION")
print("-" * 60)

print("Base learner: DecisionTreeRegressor")
print("Number of base models:", bagging_model.n_estimators)
print("Bootstrap sampling:", bagging_model.bootstrap)
print("OOB scoring enabled:", bagging_model.oob_score)
print("Parallel jobs:", bagging_model.n_jobs)

# Train the model.
bagging_model.fit(X_train, y_train)

print("\nBagging model trained successfully!")
print("Number of fitted trees:", len(bagging_model.estimators_))


------------------------------------------------------------
BAGGING MODEL CONFIGURATION
------------------------------------------------------------
Base learner: DecisionTreeRegressor
Number of base models: 100
Bootstrap sampling: True
OOB scoring enabled: False
Parallel jobs: 1

Bagging model trained successfully!
Number of fitted trees: 100


In [ ]:
# ============================================================
# 4. CALCULATE OOB SCORE AND TEST PERFORMANCE
# ============================================================

# Safely access oob_score_ only if oob_score was enabled during configuration
if getattr(bagging_model, "oob_score", False):
    oob_score = bagging_model.oob_score_
else:
    oob_score = float('nan')

# Predict prices for the held-out test observations.
bagging_predictions = bagging_model.predict(X_test)

# Calculate test metrics.
bagging_test_r2 = r2_score(y_test, bagging_predictions)

bagging_test_mse = mean_squared_error(
    y_test,
    bagging_predictions
)

print("\n" + "=" * 60)
print("BAGGING REGRESSOR RESULTS")
print("=" * 60)

if not np.isnan(oob_score):
    print(f"OOB R-squared:       {oob_score:.4f}")
else:
    print("OOB R-squared:       Not Available (OOB scoring disabled)")
print(f"Test R-squared:      {bagging_test_r2:.4f}")
print(f"Test Mean Squared Error: {bagging_test_mse:.2f}")


BAGGING REGRESSOR RESULTS
OOB R-squared:       Not Available (OOB scoring disabled)
Test R-squared:      0.5815
Test Mean Squared Error: 4185090000.00


In [ ]:
# ============================================================
# 5. TRAIN A SINGLE DECISION TREE FOR COMPARISON
# ============================================================

single_tree = DecisionTreeRegressor(random_state=42)

single_tree.fit(X_train, y_train)

single_predictions = single_tree.predict(X_test)

single_test_r2 = r2_score(y_test, single_predictions)

single_test_mse = mean_squared_error(
    y_test,
    single_predictions
)

print("\n" + "=" * 60)
print("SINGLE DECISION TREE VS BAGGING")
print("=" * 60)

print(f"{'Metric':<28}{'Single Tree':>15}{'Bagging':>15}")
print("-" * 58)

print(
    f"{'Test R-squared':<28}"
    f"{single_test_r2:>15.4f}"
    f"{bagging_test_r2:>15.4f}"
)

print(
    f"{'Test MSE':<28}"
    f"{single_test_mse:>15.2f}"
    f"{bagging_test_mse:>15.2f}"
)

print(f"\nBagging OOB R-squared: {oob_score:.4f}")


SINGLE DECISION TREE VS BAGGING
Metric                          Single Tree        Bagging
----------------------------------------------------------
Test R-squared                      -0.2250         0.5815
Test MSE                     12250000000.00  4185090000.00

Bagging OOB R-squared: nan


In [ ]:
# ============================================================
# 6. SAVE TEST PREDICTIONS TO CSV
# ============================================================

predictions_df = pd.DataFrame({
    "Actual_Price": y_test.to_numpy(),
    "Single_Tree_Prediction": single_predictions,
    "Bagging_Prediction": bagging_predictions
})

predictions_df["Bagging_Absolute_Error"] = np.abs(
    predictions_df["Actual_Price"]
    - predictions_df["Bagging_Prediction"]
)

predictions_df.to_csv("predictions.csv", index=False)

print("\nSaved: predictions.csv")


Saved: predictions.csv


In [ ]:
# ============================================================
# 7. ANALYSE MODEL STABILITY WHEN DATA CHANGES
# ============================================================

# Baseline predictions were generated using the full training set.
# Remove one training observation at a time and retrain each model.
# Compare predictions on the same fixed test set.

stability_results = []

X_train_reset = X_train.reset_index(drop=True)
y_train_reset = y_train.reset_index(drop=True)

for i in range(len(X_train_reset)):

    # Remove one training observation.
    X_reduced = X_train_reset.drop(index=i)
    y_reduced = y_train_reset.drop(index=i)

    # Train a single decision tree on the reduced dataset.
    reduced_tree = DecisionTreeRegressor(random_state=42)
    reduced_tree.fit(X_reduced, y_reduced)

    reduced_tree_predictions = reduced_tree.predict(X_test)

    tree_prediction_change = np.mean(
        np.abs(single_predictions - reduced_tree_predictions)
    )

    # Train bagging on the reduced dataset.
    # OOB is disabled here because this experiment measures
    # prediction stability, not OOB performance.
    reduced_bagging = BaggingRegressor(
        estimator=DecisionTreeRegressor(random_state=42),
        n_estimators=100,
        max_samples=1.0,
        max_features=1.0,
        bootstrap=True,
        bootstrap_features=False,
        oob_score=False,
        n_jobs=1,
        random_state=42
    )

    reduced_bagging.fit(X_reduced, y_reduced)

    reduced_bagging_predictions = reduced_bagging.predict(X_test)

    bagging_prediction_change = np.mean(
        np.abs(bagging_predictions - reduced_bagging_predictions)
    )

    stability_results.append({
        "Removed_Training_Row": i + 1,
        "Tree_Mean_Absolute_Prediction_Change": tree_prediction_change,
        "Bagging_Mean_Absolute_Prediction_Change": bagging_prediction_change
    })


stability_df = pd.DataFrame(stability_results)

print("\n" + "=" * 60)
print("MODEL STABILITY ANALYSIS")
print("=" * 60)

print(stability_df.to_string(index=False))

mean_tree_change = (
    stability_df["Tree_Mean_Absolute_Prediction_Change"].mean()
)

mean_bagging_change = (
    stability_df["Bagging_Mean_Absolute_Prediction_Change"].mean()
)

print("\nAverage prediction change after removing one training row:")
print(f"Single Decision Tree: {mean_tree_change:.2f}")
print(f"Bagging Regressor:    {mean_bagging_change:.2f}")

if mean_bagging_change < mean_tree_change:
    stability_conclusion = (
        "In this experiment, Bagging showed a smaller average "
        "prediction change than the single decision tree."
    )
elif mean_bagging_change > mean_tree_change:
    stability_conclusion = (
        "In this experiment, Bagging showed a larger average "
        "prediction change than the single decision tree."
    )
else:
    stability_conclusion = (
        "Both models showed the same average prediction change "
        "in this experiment."
    )

print("\nStability conclusion:")
print(stability_conclusion)

# Save the detailed stability results.
stability_df.to_csv("stability_analysis.csv", index=False)

print("\nSaved: stability_analysis.csv")


MODEL STABILITY ANALYSIS
 Removed_Training_Row  Tree_Mean_Absolute_Prediction_Change  Bagging_Mean_Absolute_Prediction_Change
                    1                                   0.0                                  18400.0
                    2                               65000.0                                  47000.0
                    3                              105000.0                                  54800.0
                    4                                   0.0                                  46400.0

Average prediction change after removing one training row:
Single Decision Tree: 42500.00
Bagging Regressor:    41650.00

Stability conclusion:
In this experiment, Bagging showed a smaller average prediction change than the single decision tree.

Saved: stability_analysis.csv


In [ ]:
# ============================================================
# 8. CREATE PARAMETER LOG
# ============================================================

parameter_log = f"""
TASK 2: BAGGING AND OUT-OF-BAG ERROR ANALYSIS
=============================================

Dataset: Housing Dataset
Scikit-learn Version: {sklearn.__version__}

Model Configuration
-------------------
Ensemble: BaggingRegressor
Base Estimator: DecisionTreeRegressor
Number of Base Models: {bagging_model.n_estimators}
Maximum Samples: {bagging_model.max_samples}
Maximum Features: {bagging_model.max_features}
Bootstrap Sampling: {bagging_model.bootstrap}
Bootstrap Features: {bagging_model.bootstrap_features}
OOB Score Enabled: {bagging_model.oob_score}
Number of Parallel Jobs: {bagging_model.n_jobs}
Random State: {bagging_model.random_state}

Dataset Split
-------------
Total Observations: {len(df)}
Training Observations: {len(X_train)}
Testing Observations: {len(X_test)}
Number of Features: {X_train.shape[1]}

Evaluation Results
------------------
OOB R-squared: {oob_score:.4f}
Bagging Test R-squared: {bagging_test_r2:.4f}
Bagging Test MSE: {bagging_test_mse:.2f}
Single Tree Test R-squared: {single_test_r2:.4f}
Single Tree Test MSE: {single_test_mse:.2f}

Stability Analysis
------------------
Mean Single Tree Prediction Change: {mean_tree_change:.2f}
Mean Bagging Prediction Change: {mean_bagging_change:.2f}

Conclusion
----------
{stability_conclusion}

Limitations
-----------
The dataset contains only six observations. The training and test
sets are extremely small, so OOB and test metrics are unstable.
The stability experiment is illustrative and is not sufficient
to establish general performance on unseen housing data.
"""

with open("parameter_log.txt", "w", encoding="utf-8") as file:
    file.write(parameter_log.strip())

print("Saved: parameter_log.txt")

Saved: parameter_log.txt


In [ ]:
# ============================================================
# 9. GENERATE STABILITY REPORT PDF
# ============================================================

pdf_filename = "stability_report.pdf"

document = SimpleDocTemplate(
    pdf_filename,
    pagesize=A4,
    rightMargin=45,
    leftMargin=45,
    topMargin=45,
    bottomMargin=45
)

styles = getSampleStyleSheet()

story = []

story.append(
    Paragraph(
        "Bagging and Out-of-Bag Error: Stability Report",
        styles["Title"]
    )
)

story.append(Spacer(1, 12))

story.append(
    Paragraph(
        "Objective",
        styles["Heading2"]
    )
)

story.append(
    Paragraph(
        "This experiment evaluates a Bagging Regressor using decision "
        "trees as base learners. Out-of-bag scoring estimates predictive "
        "performance using training observations left out of individual "
        "bootstrap samples. A single decision tree is used as a baseline.",
        styles["BodyText"]
    )
)

story.append(Spacer(1, 12))

story.append(
    Paragraph(
        "Model Configuration",
        styles["Heading2"]
    )
)

configuration_text = (
    f"Base estimator: DecisionTreeRegressor<br/>"
    f"Number of estimators: {bagging_model.n_estimators}<br/>"
    f"Bootstrap sampling: {bagging_model.bootstrap}<br/>"
    f"OOB scoring: {bagging_model.oob_score}<br/>"
    f"Random state: {bagging_model.random_state}<br/>"
    f"Training observations: {len(X_train)}<br/>"
    f"Testing observations: {len(X_test)}"
)

story.append(Paragraph(configuration_text, styles["BodyText"]))

story.append(Spacer(1, 12))

story.append(
    Paragraph(
        "Performance Results",
        styles["Heading2"]
    )
)

performance_data = [
    ["Metric", "Single Decision Tree", "Bagging Regressor"],
    ["Test R-squared", f"{single_test_r2:.4f}", f"{bagging_test_r2:.4f}"],
    ["Test MSE", f"{single_test_mse:.2f}", f"{bagging_test_mse:.2f}"],
    ["OOB R-squared", "Not applicable", f"{oob_score:.4f}"]
]

performance_table = Table(
    performance_data,
    colWidths=[1.7 * inch, 1.7 * inch, 1.7 * inch]
)

performance_table.setStyle(
    TableStyle([
        ("BACKGROUND", (0, 0), (-1, 0), colors.lightgrey),
        ("TEXTCOLOR", (0, 0), (-1, 0), colors.black),
        ("GRID", (0, 0), (-1, -1), 0.5, colors.grey),
        ("ALIGN", (1, 1), (-1, -1), "CENTER"),
        ("VALIGN", (0, 0), (-1, -1), "MIDDLE"),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 7),
        ("TOPPADDING", (0, 0), (-1, -1), 7)
    ])
)

story.append(performance_table)

story.append(Spacer(1, 12))

story.append(
    Paragraph(
        "Stability Under Small Data Changes",
        styles["Heading2"]
    )
)

story.append(
    Paragraph(
        "To examine sensitivity to training data, one training observation "
        "was removed at a time. Both models were retrained and their "
        "predictions were compared with the corresponding baseline "
        "predictions on the same test set. The table reports mean absolute "
        "prediction changes. Smaller changes indicate greater stability "
        "in this particular experiment.",
        styles["BodyText"]
    )
)

story.append(Spacer(1, 8))

stability_table_data = [
    [
        "Removed Row",
        "Single Tree Change",
        "Bagging Change"
    ]
]

for _, row in stability_df.iterrows():
    stability_table_data.append([
        str(int(row["Removed_Training_Row"])),
        f'{row["Tree_Mean_Absolute_Prediction_Change"]:.2f}',
        f'{row["Bagging_Mean_Absolute_Prediction_Change"]:.2f}'
    ])

stability_table = Table(
    stability_table_data,
    colWidths=[1.4 * inch, 1.8 * inch, 1.8 * inch]
)

stability_table.setStyle(
    TableStyle([
        ("BACKGROUND", (0, 0), (-1, 0), colors.lightgrey),
        ("GRID", (0, 0), (-1, -1), 0.5, colors.grey),
        ("ALIGN", (0, 0), (-1, -1), "CENTER"),
        ("VALIGN", (0, 0), (-1, -1), "MIDDLE"),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 6),
        ("TOPPADDING", (0, 0), (-1, -1), 6)
    ])
)

story.append(stability_table)

story.append(Spacer(1, 12))

story.append(
    Paragraph(
        "Interpretation",
        styles["Heading2"]
    )
)

story.append(
    Paragraph(
        stability_conclusion,
        styles["BodyText"]
    )
)

story.append(Spacer(1, 8))

story.append(
    Paragraph(
        "Limitations and Conclusion",
        styles["Heading2"]
    )
)

story.append(
    Paragraph(
        "The dataset contains only six observations, leaving just four "
        "observations for training and two for testing. Consequently, "
        "both test metrics and OOB R-squared can vary substantially and "
        "should not be treated as reliable estimates of performance on "
        "a larger housing population. The stability experiment is also "
        "illustrative rather than conclusive. A larger dataset and "
        "repeated evaluation would be needed for stronger conclusions. "
        "Bagging combines predictions from multiple bootstrap-trained "
        "decision trees; it can reduce variance, but it does not "
        "guarantee better results for every small dataset.",
        styles["BodyText"]
    )
)

document.build(story)

print("Saved: stability_report.pdf")

Saved: stability_report.pdf


In [ ]:
# ============================================================
# 10. FINAL OUTPUT SUMMARY
# ============================================================

print("\n" + "=" * 60)
print("ALL TASK 2 OUTPUTS GENERATED")
print("=" * 60)

output_files = [
    "predictions.csv",
    "parameter_log.txt",
    "stability_analysis.csv",
    "stability_report.pdf"
]

for filename in output_files:
    print(f"{filename}: {'Created' if os.path.exists(filename) else 'Not found'}")

print("\nTask 2 completed.")


ALL TASK 2 OUTPUTS GENERATED
predictions.csv: Created
parameter_log.txt: Created
stability_analysis.csv: Created
stability_report.pdf: Created

Task 2 completed.
